# PYCAVI ZP Calibration

This notebook will calibrate the swarped frame created in previous steps, so the units of the image will be in erg/s/cm$^2$ instead of e-/s. To achieve this, the notebook will download photometry of the observed field from either SDSS (astroquery.SDSS) or Skymapper (https://skymapper.anu.edu.au/) to compare and calculate the conversion factor required to obtained the desired physical units. <br>
From the final product one should be able to obtain AB magnitudes of the stars directly from the frame, as the headers store the parameters used for the photometry together with the zero points to convert to magnitudes.


## About the filters
Skymapper uses filters slightly different with respect the Sloan filters used in OmegaCam. To achieve a better comversion between both set of filters, the former ones are transformed to Sloan magnitudes using the color conversion published in SkyMapper DR1 paper, Fig 17 (https://ui.adsabs.harvard.edu/abs/2018PASA...35...10W/abstract).<br>
A table with different colors for different bandpasses is publicly available at https://skymapper.anu.edu.au/filter-transformations/, which could be use instead of the transformation from the plots (not implemented in the current version).

In the case of the NB659 filter, the calibration is based on the color-color relation for r-i and r-NB published in the VPHASp paper published by Drew et al, 2014, Table 4 (https://ui.adsabs.harvard.edu/abs/2014MNRAS.440.2036D/abstract). As VPHAS data is published in Vega system, the color-color relation was corrected with the zero points for the respective filters so it will be in AB system during the calibration process.


## About the calibration process

The calibration process is governed by the ZPCalibration class found in pycavi_zpcal.py file. The flow of the class is as follow:

- Check for the parents image to get the airmass and gain of each one. This information will be inherited by the calibrated frame.
### SDSS Calibration
- Download data from SDSS DR17 using the astroquery.SDSS class to query the STAR tables.
### Skymapper calibration
- Download the data from Skymapper with a cone search centered at the position defined by the CRVAL headers of the frame and a radius of 0.7 degrees. The raw data will be cleaned to keep only the best sources (skymapper flags and nimaflags = 0) and most likely stars (class_star > 0.9). Only the bands sm g, r, i and z will be transformed to SDSS magnitudes and saved in the final (clean) catalogue.
- Perform aperture photometry on the input frame based on the position of the stars from the clean Skymapper catalogue.
- Computes the ZP (in magnitudes) via minimization of the differences in magnitudes m$_\mathrm{sm}$ - m$_\mathrm{raw}$ in the case of the filters g, r, i and z. For NB it minimizes the differences in magnitude m$_\mathrm{r}$ - m$_\mathrm{raw}$ versus m$_\mathrm{r}$ - m$_\mathrm{i}$.
- Computes the conversion factor between e-/s (raw units) and 10**-15 erg/s/cm$^2$/arcsec$^2$ (desired physical units).
- Computes the ZP required to go directly to AB magnitudes from the calibrated frame.
- Update the headers and data of frame fits accordlingly.


## About the check of calibration

Once the calibration process of the frame is finished, the script calls SExtractor to perform photometry on the frame. Sources between Skymapper and SExtractor are cross-matched to compared their magnitudes. Check plots are created to observe how well the calibration went.

In [ ]:
import os
import sys
import glob
import logging
import numpy as np
import pandas as pd

from astropy.io import fits

sys.path.append(os.path.join(os.path.abspath(''), '..')) # to be removed in production
from pycavi import pycavi_tools

from pycavi.pycavi_zpcal import ZPCalibration, plot_calibration_check_zpcal
from pycavi import pycavi_instrument
from pycavi import pycavi_astromatic_tools
from pycavi.pycavi_astromatic_tools import get_swarp_lists_dict


In [ ]:
# set the directory folders
config_dir = os.getcwd()+'/../config/'
print(config_dir)

config = pycavi_tools.read_pycavi_config(config_dir)
instrument = config['Provenance']['instrument'].lower()
observatory = config['Provenance']['observatory'] # Needed for the atmospheric extinction curve
project_dir = config['Directories']['project_dir']
reduced_std_dir = config['Directories']['reduced_std_dir']
reduced_science_dir = config['Directories']['reduced_science_dir']  
log_dir = config['Directories']['project_logs_dir']

# Should read the instrument configuration file. This ensure
# proper keyword parsing.
#
instrument = pycavi_instrument.Instrument(config_dir=config_dir, instrument=instrument)

log_file_swarp = log_dir + "/pymeffpp_swarp.log"
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s:%(levelname)s:%(message)s",
                    handlers=[
                              logging.FileHandler(log_file_swarp),
                              logging.StreamHandler()])
logging.getLogger().setLevel(logging.INFO)

# At this point it is believed that the data reduction process
# should be closely monitored, so we calibrate target/filter sets,
# that is each data partition, one by one.
#
reject_observations = []
zpcal_dict = {}
filters = []

# We first list the standard stars fields data partitions
swarp_lists_std = get_swarp_lists_dict(reduced_std_dir, reject_observations, filters)
std_observation_tags = list(set(swarp_lists_std.keys()))
std_observation_tags.sort()                # Needed to ensure same index in every run
for i in range(len(std_observation_tags)):
    new_key = reduced_std_dir+std_observation_tags[i].split("/")[0]
    file_list = glob.glob(new_key+"/"+config['Directories']['swarp_dir_name']+'/*.coadd.fits')
    zpcal_dict[new_key] = file_list
    #logging.info("index: {} observation_tag: {}".format(i,new_key))
    #print(file_list)

# We next list the science observation fields data partitions
swarp_lists_sci = get_swarp_lists_dict(reduced_science_dir, reject_observations, filters)
sci_observation_tags = list(set(swarp_lists_sci.keys()))    
sci_observation_tags.sort()                # Needed to ensure same index in every run
for i in range(len(sci_observation_tags)):
    new_key = reduced_science_dir+sci_observation_tags[i].split("/")[0]
    file_list = glob.glob(new_key+"/"+config['Directories']['swarp_dir_name']+'/*.coadd.fits')
    zpcal_dict[new_key] = file_list
    #logging.info("index: {} observation_tag: {}".format(i,new_key))
    #print(file_list)


observation_tags = list(set(zpcal_dict.keys()))
observation_tags.sort()    # Needed to ensure same index in every run
images_to_calibrate = []
for i in range(len(observation_tags)):
    logging.debug("index: {} observation_tag: {}".format(i, observation_tags[i]))
    for j in range(len(zpcal_dict[observation_tags[i]])):
        images_to_calibrate.append(zpcal_dict[observation_tags[i]][j])

for i in range(len(images_to_calibrate)):
    logging.info("index: {} -- file: {}".format(i, images_to_calibrate[i]))



In [ ]:
# Here we do the calibration of each file
#

mag_min = 10.5  # minimum magnitude to consider from skymapper catalogue when determining zero point
mag_max = 19.5  # maximum magnitude to consider from skymapper catalogue when determining zero point
aperture_radius = 5.0  # aperture in arcsec when performing aperture photometry on frame 
ap_sky_r1 = 6.0 # annulus inner radius to determine background around stars during aperture photometry
ap_sky_r2 = 9.0 # annulus outer radius to determine background around stars during aperture photometry
query_calibrator = "Skymapper"  # SDSS for northern or Skymapper for southern latitudes
query_radius=0.7
query_min_mag=9.0
query_max_mag=19.0

# See list above to select a data partition to calibrate
# For example, if the list is:
#
# index: 0 -- file: /Users/fselman/Pictures/OCAM/MultiObject/science/CP_Pup/Swarp_dir/CP_Pup_r_SDSS.coadd.fits
# index: 1 -- file: /Users/fselman/Pictures/OCAM/MultiObject/science/M67/Swarp_dir/M67_g_SDSS.coadd.fits
# index: 2 -- file: /Users/fselman/Pictures/OCAM/MultiObject/science/M67/Swarp_dir/M67_r_SDSS.coadd.fits
# index: 3 -- file: /Users/fselman/Pictures/OCAM/MultiObject/science/M67/Swarp_dir/M67_NB_659.coadd.fits
# index: 4 -- file: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/Swarp_dir/SA101_g_SDSS.coadd.fits
# index: 5 -- file: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/Swarp_dir/SA101_r_SDSS.coadd.fits
# index: 6 -- file: /Users/fselman/Pictures/OCAM/MultiObject/std/SA98/Swarp_dir/SA98_NB_659.coadd.fits
#
# then to calibrate M67_NB_659.coadd.fits set
#
# idx_selected = [3]
#
idx_selected = []
for i in range(len(images_to_calibrate)):

    # skip indices not in list idx_selected
    if not (i in idx_selected or not idx_selected):
        continue
    image = images_to_calibrate[i]
    swarp_dir = os.path.dirname(image)
    calibration_dir = swarp_dir+'/../'+config['Directories']['cal_dir_name']+'/'
    print(swarp_dir)
    logging.info("Starting to work on frame {}".format(image.split("/")[-1]))

    # This step queries a photometric database to extract positions and magnitudes
    # to be used to set the zero point of the frame. Currently only skymapper and SDSS
    # are used. Skymapper covers the southern skies and SDSS the northern one.
    #
    zpcal = ZPCalibration(image, swarp_dir, calibration_dir, observatory, instrument,
                          mag_min=mag_min, mag_max=mag_max, 
                          ap_radius=aperture_radius, ap_sky_r1=ap_sky_r1, ap_sky_r2=ap_sky_r2,
                          query_calibrator=query_calibrator, 
                          query_radius=query_radius, 
                          query_min_mag=query_min_mag, query_max_mag=query_max_mag)
    logging.info("zpcal.calibrationdir: {}".format(zpcal.calibrationdir))
    os.chdir(zpcal.calibrationdir)

    # Frame has been updated. Now we should use magzp top read magnitudes directly
    # from frame
    logging.info("zpcal.frame: {}".format(zpcal.frame))

    redo_sextractor = True
    ready_to_sextract = pycavi_astromatic_tools.prepare_to_sextract(zpcal.swarpdir + "/"+zpcal.frame, redo_sextractor)

    # we tune some Sextractor parameters
    frame_hdr = fits.getheader(image)
    mag_zp = frame_hdr["HIERARCH ESO PYCAVI QC MAGZP"]
    special_pars = {}
    special_pars["CATALOG_TYPE"] = "ASCII_HEAD"    # Needs ascii catalogue here
    special_pars["DETECT_MINAREA"] = 5              # minimum number of pixels above threshold
    special_pars["DETECT_MAXAREA"] = 0              # maximun number of pixels above threshold
    special_pars["DETECT_THRESH"] = 3             # <sigmas> or <threshold>,<ZP> in mag.arcsec-2
    special_pars["ANALYSIS_THRESH"] = 5           # <sigmas> or <threshold>,<ZP> in mag.arcsec-2
    special_pars["SEEING_FWHM"] = 1.2               # stellar FWHM in arcsec
    special_pars["PHOT_APERTURES"] = 20             # * MAG_APER aperture diameter(s) in pixels
    special_pars["PHOT_PETROPARAMS"] =  [2.0, 3.5]
    special_pars["BACKPHOTO_TYPE"] = "LOCAL"
    special_pars["MAG_ZEROPOINT"] =  mag_zp         # magnitude zero-point
    special_pars["PARAMETERS_LIST"] = ["NUMBER", "ALPHA_J2000", "DELTA_J2000", 
                                      "FLUX_BEST", "FLUXERR_BEST", "MAG_BEST", "MAGERR_BEST",
                                      "FLAGS", "FWHM_IMAGE", "ELLIPTICITY", "CLASS_STAR"]
    # Needed by zpcal and plotting
    #   1 NUMBER                 Running object number                                     
    #   2 ALPHA_J2000            Right ascension of barycenter (J2000)                      [deg]
    #   3 DELTA_J2000            Declination of barycenter (J2000)                          [deg]
    #   4 FLUX_BEST              Best of FLUX_AUTO and FLUX_ISOCOR                          [count]
    #   5 FLUXERR_BEST           RMS error for BEST flux                                    [count]
    #   6 MAG_BEST               Best of MAG_AUTO and MAG_ISOCOR                            [mag]
    #   7 MAGERR_BEST            RMS error for MAG_BEST                                     [mag]
    #   8 FLAGS                  Extraction flags                                          
    #   9 FWHM_IMAGE             FWHM assuming a gaussian core                              [pixel]
    #  10 ELLIPTICITY            1 - B_IMAGE/A_IMAGE                                       
    #  11 CLASS_STAR             S/G classifier output                                     

    sextractor_dir = swarp_dir+'/../'+config['Directories']['sex_dir_name']+'/'
    if ready_to_sextract:
        pycavi_astromatic_tools.run_sextractor_on_target(zpcal.swarpdir + "/"+zpcal.frame, sextractor_dir, special_pars)
    
    #run_sextractor_on_frame(zpcal.swarpdir + "/"+zpcal.frame, swarp_dir)
    
    phot_catalogue = "{0}_query_result_{1}_clean.csv".format(query_calibrator.lower(), zpcal.object)
    logging.info("phot_catalogue_path: {}".format(phot_catalogue))
    sex_catalogue_path = zpcal.swarpdir+'/'+zpcal.frame[:-5]+".cat"
    logging.info("sex_catalogue_path: {}".format(sex_catalogue_path))
    plot_calibration_check_zpcal(phot_catalogue, sex_catalogue_path, zpcal)
    logging.info("Frame {} successfully calibrated".format(zpcal.frame))
    
logging.info("-------Done-------")